<h2>File paths and imports</h2>

This steps is important because it tells the program where it can access the files needed throughout the process.

You must specify 3 paths:

<li> <b>model_path</b>: body detection model
<li> <b>input_video_directory</b>: Folder containing the input videos
<li> <b>output_video_directory</b>: Folder where the treated videos will be redirected to

You can also mention videos that you don't want to be treated. To do so, simply indicate their name(s) without the extension in <b>ignore_S1</b> and <b>ignore_S2</b>.

In [1]:
from ui_lib import *

# path of the body detection model
model_path = "Body_detection_model.pt"

# video paths:
# input video directory (without any annotation)
input_video_directory = "input/"
# output (final version - with human interaction)
output_video_directory = "output/"

ignore_S1 = [ # related to step 1
    "example_1",
    "example_2",
    "20241009 - 09h07.MP4",
    "20241009 - 09h07.MP4",
    "20241009 - 09h07",
    #"20241015 - 12h41-(tempCut)",
    "big.MP4",
    "big"
]

ignore_S2 = [ # related to step 2
    "example_1",
    "example_2",
    "20241009 - 09h07.MP4",
    "20241009 - 09h07",
    "big.MP4",
    "big"
]


ignore_S3 = [ # related to step 3
    "example_1",
    "example_2",
    "20241009 - 09h07.MP4",
    "20241009 - 09h07"
]

/home/diego/.pyenv/versions/chimprec310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/diego/.pyenv/versions/chimprec310/lib/python3.10/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


<h2>First step:</h2>

This step will process the input videos automatically. In other words, it will draw rectangles around each individuals and track them throughout the video.

In [2]:
#FIRST STEP CODE (double-click to extend)

# directory containing all the manual modifications
mannual_annotations_directory = f"{input_video_directory}/manual_annotations"
output_video_directory_temp = f"{output_video_directory}/temp"
raw_text_output_directory = f"{output_video_directory_temp}/raw_output"

# Create the directories if they do not exist yet
os.makedirs(input_video_directory, exist_ok=True)
os.makedirs(output_video_directory, exist_ok=True)
os.makedirs(mannual_annotations_directory, exist_ok=True)
os.makedirs(output_video_directory_temp, exist_ok=True)
os.makedirs(raw_text_output_directory, exist_ok=True)

max_cosine_distance = 0.5       # maximal distance to match an object (lower = more strict)
nn_budget = None                # maximal buffer size
metric = nn_matching.NearestNeighborDistanceMetric("cosine", max_cosine_distance, nn_budget)

# YOLOv8s initialisation
YOLOv8s = YOLO(model_path)

# DeepSORT initialisation
DeepSort = DeepSortTracker(metric)

# Osnet initialisation
Osnet = torchreid.models.build_model(name='osnet_x1_0', num_classes=751, pretrained=True)
Osnet.eval()

for input_video in os.listdir(input_video_directory):
    if input_video.endswith(".mp4") or input_video.endswith(".MP4"):

        full_video_path = os.path.join(input_video_directory, input_video)
        video_name = os.path.splitext(input_video)[0]

        if video_name in ignore_S1:
            print(f"{video_name}.mp4 ignored")
            continue

        # creation of the manual_annotation textfile if it doesn't exist yet
        annotation_file_path = f"{mannual_annotations_directory}/{video_name}.txt"
        try:
            with open(annotation_file_path, 'x') as f:
                print(f"{video_name}.txt automatically created in {mannual_annotations_directory}.")
        except FileExistsError:
            print(f"{video_name}.txt already present in {mannual_annotations_directory}.")
        print("\n")

        # production of the textual outputs
        perform_tracking(
            input_video_path = full_video_path, 
            output_text_file_path = f"{raw_text_output_directory}/{video_name}.txt", 
            detection_model = YOLOv8s, 
            tracker = DeepSort,
            confidence_threshold = 0.5, 
            model_feature_extraction = Osnet
        )
        print(f"Annotations ready for video: {full_video_path}.\n")

        ######## AUDIO PREP ########
        processed_no_audio = f"{output_video_directory_temp}/{video_name}-(temp).mp4"
        processed_with_audio = f"{output_video_directory_temp}/{video_name}-(temp)-audio.mp4"

        # production of the visual output
        draw_bbox_from_file(
            file_path = f"{raw_text_output_directory}/{video_name}.txt", 
            input_video_path = full_video_path, 
            output_video_path = processed_no_audio,
            annotation_type="bbox",
            draw_frame_count=True
        )

        print("Adding audio...")
        ######## AUDIO ########
        mux_audio(full_video_path, processed_no_audio, processed_with_audio)


        print(f"Treatment done: {full_video_path}.\n")

Successfully loaded imagenet pretrained weights from "/home/diego/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
20241009 - 09h07.mp4 ignored
20241015 - 12h41-(tempCut).txt already present in input//manual_annotations.




Tracking progress (20241015 - 12h41-(tempCut).mp4): 100%|██████████| 484/484 [00:55<00:00,  8.77it/s]


Annotations ready for video: input/20241015 - 12h41-(tempCut).mp4.



Drawing annotations (20241015 - 12h41-(tempCut).mp4): 100%|██████████| 484/484 [00:09<00:00, 50.19it/s]


Adding audio...
Treatment done: input/20241015 - 12h41-(tempCut).mp4.

big.mp4 ignored


ffmpeg version n8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.1 (GCC) 20251112
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enab

<h2>Second step:</h2>

This final step will take into account your modifications to modify the output of the automated process.

<b>If you need to modify annotations previously created:</b> simply run this part of the code. In this case, there's no need to run the above cells.

In [4]:
#SECOND STEP CODE (double-click to extend)

for input_video in os.listdir(input_video_directory):
    if input_video.endswith(".mp4") or input_video.endswith(".MP4"):
        full_video_path = os.path.join(input_video_directory, input_video)
        video_name = os.path.splitext(input_video)[0]

        if video_name in ignore_S2:
            print(f"{video_name}.mp4 ignored")
            continue

        annotation_file = f"{mannual_annotations_directory}/{video_name}.txt"

        raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

        try:
            edit_reader = modification_reader(annotation_file)
        except:
            print(f"Error: the manual annotation file related to the video <{full_video_path}> is not found. It must be located at <{annotation_file}>.")
            continue
        
        metadata_file_path = f"{output_video_directory}/{video_name}-treated.txt"
        output_video_path = f"{output_video_directory}/{video_name}-treated.mp4"
        writer = data_writer(metadata_file_path)

        # computation of the new metadata file
        modified_data = edit_raw_output(raw_reader, edit_reader) 

        # production of the textual output
        writer.write(modified_data)


        processed_no_audio = f"{output_video_directory}/{video_name}-treated.mp4"
        processed_with_audio = f"{output_video_directory}/{video_name}-treated-audio.mp4"

        # production of the visual output
        draw_bbox_from_file(
            file_path = metadata_file_path, 
            input_video_path = full_video_path, 
            output_video_path = processed_no_audio,
            annotation_type="bbox",
            draw_frame_count=True
        )

        print("Adding audio...")
        
        ######## AUDIO ########
        mux_audio(full_video_path, processed_no_audio, processed_with_audio)

        print(f"Treatment done: {full_video_path}.\n")


### Could add a final step that just draw the arrows and the names when the manual annotation process is fully done.

20241009 - 09h07.mp4 ignored


Drawing annotations (20241015 - 12h41-(tempCut).mp4): 100%|██████████| 484/484 [00:10<00:00, 47.33it/s]


Adding audio...
Treatment done: input/20241015 - 12h41-(tempCut).mp4.

big.mp4 ignored


ffmpeg version n8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.1 (GCC) 20251112
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enab

<h2> Third Step: </h2>

This final step will only be used to generate the arrows associated with the corresponding chimpanzees.<br>
<b>It's only to be performed when the manual annotations are 100% correct.</b>

In [5]:
# just draw the arrows and the names when the manual annotation process is fully done.
for input_video in os.listdir(input_video_directory):
    if input_video.endswith(".mp4") or input_video.endswith(".MP4"):
        full_video_path = os.path.join(input_video_directory, input_video)
        video_name = os.path.splitext(input_video)[0]

        if video_name in ignore_S2:
            print(f"{video_name}.mp4 ignored")
            continue

        annotation_file = f"{mannual_annotations_directory}/{video_name}.txt"

        raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

        try:
            edit_reader = modification_reader(annotation_file)
        except:
            print(f"Error: the manual annotation file related to the video <{full_video_path}> is not found. It must be located at <{annotation_file}>.")
            continue
        
        metadata_file_path = f"{output_video_directory}/{video_name}-treated.txt"
        output_video_path = f"{output_video_directory}/{video_name}-final.mp4"
        writer = data_writer(metadata_file_path)

        # computation of the new metadata file
        modified_data = edit_raw_output(raw_reader, edit_reader) 

        # production of the textual output
        writer.write(modified_data)

        processed_no_audio = f"{output_video_directory}/{video_name}-final.mp4"
        processed_with_audio = f"{output_video_directory}/{video_name}-final-audio.mp4"

        # production of the visual output
        draw_bbox_from_file(
            file_path = metadata_file_path, 
            input_video_path = full_video_path, 
            output_video_path = processed_no_audio,
            annotation_type="triangle",
            draw_frame_count=False
        )
        print("Adding audio...")
        mux_audio(full_video_path, processed_no_audio, processed_with_audio)
        print(f"Treatment done: {full_video_path}.\n")



20241009 - 09h07.mp4 ignored


Drawing annotations (20241015 - 12h41-(tempCut).mp4): 100%|██████████| 484/484 [00:12<00:00, 38.39it/s]


Adding audio...
Treatment done: input/20241015 - 12h41-(tempCut).mp4.

big.mp4 ignored


ffmpeg version n8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.1 (GCC) 20251112
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enab